# DS2002 · Reflection What Broke What Mattered

**Studio — 2026-11-04 · Fall 2026**  
**Class time:** 45 minutes

---

## Reflection — what broke, what mattered

The project is submitted. This hour is for extracting what is reusable from it, because the capstone starts in under two weeks and it is the same work at larger scale.

Write individually for the first fifteen minutes, then compare with your team. The disagreements are the interesting part — if two people on the same team have different accounts of which decision mattered most, at least one of you learned something the other did not.

Answers here should be specific enough that someone could act on them. "Communicate better" is not usable. "Name one notebook owner per week, because we spent two hours reconciling two versions of the cleaning" is.

### 1. Data quality

Which single data-quality problem, if it had gone unnoticed, would have changed your conclusion the most? Give the number both ways.

The SKU fragmentation.
Strawberry Pop-Tarts was sold under 4 SKUs (PT-12, SKU#459812, POPTART-STRAW, PT_STRAWBERRY).
**Unfixed:** its biggest SKU is 1,332 units and ranks #8 of 45 SKUs, and the other three rank #14, #17 and #18, so Pop-Tarts never shows up in any top-5 list.
**Fixed:** it is 5,005 units, the #2 product of 20, 32 units behind bottled water, and at 9.2x baseline it had the biggest surge of anything in the store.
Without the consolidation our headline finding (and recommendation 3, "plan Pop-Tarts as a storm item") would not exist, and the SKU top 5 would have shown water three times instead.

### 2. ETL trade-offs

Name a cleaning decision where a reasonable person could have chosen the other way. What was the other option, what would it have done to your reported numbers, and would you make the same call again?

Dropping the 881 rows with a blank, `NULL`, `N/A` or `NaN` timestamp from the whole dataset.
The other option was to drop them only from the date-based questions (Q1, Q3, Q4) and keep them for the totals in Q2, since their store, SKU and quantity were fine.

Those rows hold 1,687 units, 3.4% of all units.
Keeping them in Q2 would have raised the totals (water 5,037 to 5,227, Pop-Tarts 5,005 to 5,198) but not changed the ranking, and Q1 cannot use undated rows either way.
The risk with keeping them is that Q2 and Q1 would then be computed on different row sets, so their totals would stop reconciling.

I would make the same call, but I would put the 3.4% in the Q2 finding so nobody reads 5,005 as the complete count.
For comparison, the other judgment call (dropping 130 negative quantities as returns instead of netting them) only moves the Q1 multiple from 2.8761x to 2.8764x, because the 237 returned units are under 0.5% of everything sold.

### 3. Pipeline trust

If someone handed you this notebook and asked whether the numbers are right, what would you check first? Is that check currently in the notebook? If not, that is your first capstone improvement.

First, that every raw row is accounted for: 25,562 raw rows minus 30 duplicates, 130 negative quantities and 881 undated rows is exactly the 24,521 clean rows.
The decision log has each of those numbers, but nothing in the notebook asserts that they add up, so a future cleaning step could silently drop rows and the log would still look fine.
That `assert len(raw) - sum(dropped) == len(clean)` is my first capstone improvement.

Second, the cross-check against the inventory database.
The notebook shows the two systems move together (r = 0.991), but the database runs at a steady 5.1x the transaction file, and we never explained why.
A reviewer would ask, and right now our answer is "we don't know".

### 4. Business versus data

Your recommendation went past what the data strictly proves — every useful recommendation does. Where exactly is that line in your work, and how would you say it out loud to someone who is about to spend money based on it?

What the data proves: during Helene, at these 12 stores, units ran 2.9x normal on Sep 21-25, peaked 1-2 days before landfall, and the surge was about the same (2.6x-3.2x) at every store regardless of distance.
Where we went past it: "stock every Florida store at least 5 days before landfall" and "plan Pop-Tarts as a storm item" assume the next storm looks like this one.
We have one storm, so the 5 days, the 3x and the 9.2x are each one observation.

How I would say it to someone about to spend money:
"In Helene, demand roughly tripled statewide starting about 5 days out, and Pop-Tarts moved the most.
That is one storm, so use it to set the timing and the store list, but check the volumes against Ian and Idalia before committing the full order."

### 5. Team process

What cost your team the most time that was not a data problem? Merge conflicts, duplicated work, waiting on someone, unclear ownership? Name the specific change you will make for the capstone.

Keeping the team notebook runnable for everyone, not the data.
When I went to push the progress lab on Nov 1, my push was rejected because the team notebook had been updated the day before.
That update added a `seaborn` import that was not installed on my machine, and because my notebook runs the team notebook, it failed partway through without an obvious error until I checked.
The diff for that one update was over 8,000 lines, almost all saved cell outputs and images, so nobody could have reviewed it anyway.
Earlier, on Oct 28, we also ended up with a merge commit carrying git's default unedited message because two of us had diverged on `main`.

For the capstone: a `requirements.txt` in the repo from day one, outputs cleared before every commit (or `nbstripout` as a git filter) so diffs show only code, and a short heads-up in the group chat before anyone pushes changes to the shared notebook.

### 6. Carry-forward list

**TODO:** three concrete things you are taking into the capstone. Code you will reuse, a habit you are adopting, or a mistake you are not repeating.

Run the cell — this list is your capstone kickoff agenda, so make it real.

In [1]:
carry_forward = [
    'Reuse: the log(step, decision, rows) decision log and get_with_retries() from the team notebook, plus profile() for the first look at every raw file',
    'Habit: assert row counts and unit totals at every join and consolidation (validate="one_to_one", units before == units after), and assert raw rows minus logged drops == clean rows',
    'Not repeating: undeclared dependencies and committed notebook outputs; requirements.txt and stripped outputs from the first commit',
]

for i, item in enumerate(carry_forward, 1):
    flag = 'unfilled' if item.startswith('TODO') else 'ok'
    print(f'{i}. [{flag}] {item}')
print()
print('bring this list to the capstone kickoff on Nov 16.')

1. [ok] Reuse: the log(step, decision, rows) decision log and get_with_retries() from the team notebook, plus profile() for the first look at every raw file
2. [ok] Habit: assert row counts and unit totals at every join and consolidation (validate="one_to_one", units before == units after), and assert raw rows minus logged drops == clean rows
3. [ok] Not repeating: undeclared dependencies and committed notebook outputs; requirements.txt and stripped outputs from the first commit

bring this list to the capstone kickoff on Nov 16.


---

## Checkpoint (participation)

Report the single data-quality problem that mattered most and the number it would have changed.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [2]:
# Checkpoint
problem_that_mattered = 'Strawberry Pop-Tarts split across 4 SKUs (PT-12, SKU#459812, POPTART-STRAW, PT_STRAWBERRY)'
number_with_fix = 'Pop-Tarts 5,005 units, #2 of 20 products, 9.2x surge'
number_without_fix = 'largest Pop-Tarts SKU 1,332 units, #8 of 45 SKUs, in no top-5 list'
capstone_change = 'requirements.txt and stripped notebook outputs from the first commit, so a teammate update cannot silently break the run'

print('problem:', problem_that_mattered)
print('reported:', number_with_fix, 'vs unfixed:', number_without_fix)
print('changing for the capstone:', capstone_change)

problem: Strawberry Pop-Tarts split across 4 SKUs (PT-12, SKU#459812, POPTART-STRAW, PT_STRAWBERRY)
reported: Pop-Tarts 5,005 units, #2 of 20 products, 9.2x surge vs unfixed: largest Pop-Tarts SKU 1,332 units, #8 of 45 SKUs, in no top-5 list
changing for the capstone: requirements.txt and stripped notebook outputs from the first commit, so a teammate update cannot silently break the run
